# Accusation target annotation
This notebook creates a version of the transcript

In [12]:
from pathlib import Path
from typing import Any
import json
import shutil


INDEX_PATH = (
    REPO_ROOT
    / "data"
    / "processed"
    / "lai2023"
    / "onuw_transcripts_ready"
    / "index_cleaned.json"
)

ANNOTATIONS_ROOT = REPO_ROOT / "data" / "lai2023"

OUTPUT_ROOT = (
    REPO_ROOT
    / "data"
    / "processed"
    / "lai2023"
    / "accusation_transcripts"
    / "ready_for_annotation"
)

ACCUSATION_MARKER = "<<ACCUSATION_TO_RESOLVE>>"


def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def encode_youtube_session_name(raw_session_name: Any) -> str:
    return str(raw_session_name).strip().replace(" ", "#")


def get_annotation_key(
    game: dict[str, Any],
    source: str,
) -> tuple[str, str, str]:
    if source == "Youtube":
        session_name = encode_youtube_session_name(
            game["video_name"]
        )
    elif source == "Ego4D":
        session_name = str(game["EG_ID"]).strip()
    else:
        raise ValueError(f"Unknown source: {source!r}")

    game_key = str(game["Game_ID"]).strip()

    return source, session_name, game_key


def create_accusation_transcripts() -> None:
    """
    Create transcripts directly from the annotation JSON files.

    Only games present in index_cleaned.json are retained.

    Output:
        accusation_transcripts/
            Youtube/
            Ego4D/
    """
    index_items = load_json(INDEX_PATH)

    indexed_games = {
        (
            str(item["source"]).strip(),
            str(item["session_name"]).strip(),
            str(item["game_key"]).strip(),
        ): item
        for item in index_items
    }

    if OUTPUT_ROOT.exists():
        shutil.rmtree(OUTPUT_ROOT)

    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    written_games = 0
    games_without_accusations = 0
    total_accusations = 0
    found_index_keys = set()

    for source in ("Youtube", "Ego4D"):
        split_dir = ANNOTATIONS_ROOT / source / "split"
        output_source_dir = OUTPUT_ROOT / source
        output_source_dir.mkdir(parents=True, exist_ok=True)

        for annotation_path in sorted(split_dir.glob("*.json")):
            games = load_json(annotation_path)

            for game in games:
                key = get_annotation_key(
                    game=game,
                    source=source,
                )

                # Keep only games listed in index_cleaned.json.
                if key not in indexed_games:
                    continue

                if key in found_index_keys:
                    raise ValueError(
                        f"Duplicate indexed game in annotations: {key}"
                    )

                found_index_keys.add(key)

                _, session_name, game_key = key

                dialogue = game.get("Dialogue", [])
                player_names = game.get(
                    "playerNames",
                    indexed_games[key].get("player_names", []),
                )

                n_accusations = sum(
                    "Accusation" in row.get("annotation", [])
                    for row in dialogue
                )

                if n_accusations == 0:
                    games_without_accusations += 1
                    continue

                output_lines = [
                    f"Source: {source}",
                    f"Session: {session_name}",
                    f"Game: {game_key}",
                    f"Players: {', '.join(player_names)}",
                    "",
                    "Transcript:",
                ]

                for row in dialogue:
                    rec_id = row["Rec_Id"]
                    speaker = str(row.get("speaker", "")).strip()
                    utterance = str(
                        row.get("utterance", "")
                    ).strip()

                    line = f"[{rec_id}] {speaker}: {utterance}"

                    if "Accusation" in row.get("annotation", []):
                        line += f" {ACCUSATION_MARKER}"
                        total_accusations += 1

                    output_lines.append(line)

                output_path = (
                    output_source_dir
                    / f"{session_name}_{game_key}.txt"
                )

                output_path.write_text(
                    "\n".join(output_lines),
                    encoding="utf-8",
                )

                written_games += 1

    missing_index_games = set(indexed_games) - found_index_keys

    print("Indexed games:", len(indexed_games))
    print("Written games:", written_games)
    print(
        "Indexed games without accusations:",
        games_without_accusations,
    )
    print("Marked accusation rows:", total_accusations)
    print(
        "Indexed games missing from annotation files:",
        len(missing_index_games),
    )

    if missing_index_games:
        print("\nMissing games:")
        for key in sorted(missing_index_games):
            print(key)


create_accusation_transcripts()

Indexed games: 191
Written games: 190
Indexed games without accusations: 1
Marked accusation rows: 3499
Indexed games missing from annotation files: 0


In [13]:
from pathlib import Path
from typing import Any
import json

import pandas as pd


def validate_accusation_transcripts(
    index_path: Path = INDEX_PATH,
    annotations_root: Path = ANNOTATIONS_ROOT,
    output_root: Path = OUTPUT_ROOT,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Validate the accusation transcript files.

    Expected behavior:
    - index_cleaned.json defines the games to retain;
    - transcript content comes directly from annotation Dialogue rows;
    - rows containing 'Accusation' receive ACCUSATION_MARKER;
    - games without accusations have no output file.
    """

    index_items = load_json(index_path)

    indexed_games = {
        (
            str(item["source"]).strip(),
            str(item["session_name"]).strip(),
            str(item["game_key"]).strip(),
        ): item
        for item in index_items
    }

    if len(indexed_games) != len(index_items):
        raise ValueError(
            "index_cleaned.json contains duplicate "
            "(source, session_name, game_key) entries."
        )

    annotation_games = {}

    # -------------------------------------------------------------
    # Collect indexed annotation games
    # -------------------------------------------------------------
    for source in ("Youtube", "Ego4D"):
        split_dir = annotations_root / source / "split"

        for annotation_path in sorted(split_dir.glob("*.json")):
            games = load_json(annotation_path)

            for game in games:
                key = get_annotation_key(
                    game=game,
                    source=source,
                )

                if key not in indexed_games:
                    continue

                if key in annotation_games:
                    raise ValueError(
                        "Duplicate indexed game in annotation files:\n"
                        f"{key}\n"
                        f"First file: "
                        f"{annotation_games[key]['annotation_file']}\n"
                        f"Second file: {annotation_path}"
                    )

                annotation_games[key] = {
                    "game": game,
                    "annotation_file": str(annotation_path),
                }

    validation_records = []
    expected_output_paths = set()

    # -------------------------------------------------------------
    # Validate every indexed game
    # -------------------------------------------------------------
    for key, index_item in indexed_games.items():
        source, session_name, game_key = key

        output_path = (
            output_root
            / source
            / f"{session_name}_{game_key}.txt"
        )

        if key not in annotation_games:
            validation_records.append({
                "source": source,
                "session_name": session_name,
                "game_key": game_key,
                "status": "missing_annotation_game",
                "n_dialogue_rows": None,
                "n_expected_accusations": None,
                "n_actual_markers": None,
                "missing_marker_ids": None,
                "extra_marker_ids": None,
                "output_path": str(output_path),
            })
            continue

        game = annotation_games[key]["game"]
        dialogue = game.get("Dialogue", [])

        expected_accusation_ids = [
            int(row["Rec_Id"])
            for row in dialogue
            if "Accusation" in row.get("annotation", [])
        ]

        n_expected_accusations = len(expected_accusation_ids)

        # Games without accusations should not have an output file.
        if n_expected_accusations == 0:
            status = (
                "unexpected_file_for_game_without_accusations"
                if output_path.exists()
                else "no_accusations_ok"
            )

            validation_records.append({
                "source": source,
                "session_name": session_name,
                "game_key": game_key,
                "status": status,
                "n_dialogue_rows": len(dialogue),
                "n_expected_accusations": 0,
                "n_actual_markers": 0,
                "missing_marker_ids": [],
                "extra_marker_ids": [],
                "output_path": str(output_path),
            })
            continue

        expected_output_paths.add(output_path.resolve())

        if not output_path.exists():
            validation_records.append({
                "source": source,
                "session_name": session_name,
                "game_key": game_key,
                "status": "missing_output_file",
                "n_dialogue_rows": len(dialogue),
                "n_expected_accusations": n_expected_accusations,
                "n_actual_markers": 0,
                "missing_marker_ids": expected_accusation_ids,
                "extra_marker_ids": [],
                "output_path": str(output_path),
            })
            continue

        # ---------------------------------------------------------
        # Reconstruct exactly what the output should contain
        # ---------------------------------------------------------
        player_names = game.get(
            "playerNames",
            index_item.get("player_names", []),
        )

        expected_lines = [
            f"Source: {source}",
            f"Session: {session_name}",
            f"Game: {game_key}",
            f"Players: {', '.join(player_names)}",
            "",
            "Transcript:",
        ]

        expected_marker_ids = []
        expected_dialogue_lines = {}

        for row in dialogue:
            rec_id = int(row["Rec_Id"])
            speaker = str(row.get("speaker", "")).strip()
            utterance = str(row.get("utterance", "")).strip()

            line = f"[{rec_id}] {speaker}: {utterance}"

            if "Accusation" in row.get("annotation", []):
                line += f" {ACCUSATION_MARKER}"
                expected_marker_ids.append(rec_id)

            expected_lines.append(line)
            expected_dialogue_lines[rec_id] = line

        expected_text = "\n".join(expected_lines)
        actual_text = output_path.read_text(
            encoding="utf-8"
        )

        # Ignore only a final newline difference.
        exact_content_match = (
            actual_text.rstrip("\n")
            == expected_text.rstrip("\n")
        )

        # ---------------------------------------------------------
        # Read actual marker IDs
        # ---------------------------------------------------------
        actual_marker_ids = []

        for line in actual_text.splitlines():
            if ACCUSATION_MARKER not in line:
                continue

            if not line.startswith("[") or "]" not in line:
                continue

            rec_id_text = line[1:line.index("]")]

            try:
                actual_marker_ids.append(
                    int(rec_id_text)
                )
            except ValueError:
                pass

        missing_marker_ids = sorted(
            set(expected_marker_ids)
            - set(actual_marker_ids)
        )

        extra_marker_ids = sorted(
            set(actual_marker_ids)
            - set(expected_marker_ids)
        )

        duplicate_marker_ids = sorted({
            rec_id
            for rec_id in actual_marker_ids
            if actual_marker_ids.count(rec_id) > 1
        })

        if not exact_content_match:
            status = "content_mismatch"
        elif missing_marker_ids:
            status = "missing_markers"
        elif extra_marker_ids:
            status = "extra_markers"
        elif duplicate_marker_ids:
            status = "duplicate_markers"
        elif len(actual_marker_ids) != n_expected_accusations:
            status = "marker_count_mismatch"
        else:
            status = "ok"

        validation_records.append({
            "source": source,
            "session_name": session_name,
            "game_key": game_key,
            "status": status,
            "n_dialogue_rows": len(dialogue),
            "n_expected_accusations": n_expected_accusations,
            "n_actual_markers": len(actual_marker_ids),
            "missing_marker_ids": missing_marker_ids,
            "extra_marker_ids": extra_marker_ids,
            "duplicate_marker_ids": duplicate_marker_ids,
            "exact_content_match": exact_content_match,
            "output_path": str(output_path),
        })

    validation_df = pd.DataFrame(validation_records)

    # -------------------------------------------------------------
    # Check for unexpected .txt files
    # -------------------------------------------------------------
    actual_output_paths = {
        path.resolve()
        for source in ("Youtube", "Ego4D")
        for path in (output_root / source).glob("*.txt")
    }

    unexpected_paths = sorted(
        actual_output_paths - expected_output_paths
    )

    unexpected_files_df = pd.DataFrame({
        "unexpected_output_file": [
            str(path)
            for path in unexpected_paths
        ]
    })

    # -------------------------------------------------------------
    # Summary
    # -------------------------------------------------------------
    status_counts = (
        validation_df["status"]
        .value_counts(dropna=False)
        .rename_axis("status")
        .reset_index(name="n_games")
    )

    print("VALIDATION STATUS")
    display(status_counts)

    print("FILES")
    print(
        "Expected output files:",
        len(expected_output_paths),
    )
    print(
        "Actual output files:",
        len(actual_output_paths),
    )
    print(
        "Unexpected output files:",
        len(unexpected_paths),
    )

    print("\nACCUSATIONS")
    print(
        "Expected accusation markers:",
        int(
            validation_df[
                "n_expected_accusations"
            ]
            .fillna(0)
            .sum()
        ),
    )
    print(
        "Actual accusation markers:",
        int(
            validation_df[
                "n_actual_markers"
            ]
            .fillna(0)
            .sum()
        ),
    )

    problematic_df = validation_df[
        ~validation_df["status"].isin([
            "ok",
            "no_accusations_ok",
        ])
    ].copy()

    print("\nPROBLEMATIC GAMES:", len(problematic_df))
    display(problematic_df)

    if unexpected_paths:
        print("\nUNEXPECTED OUTPUT FILES")
        display(unexpected_files_df)

    if (
        problematic_df.empty
        and not unexpected_paths
    ):
        print(
            "\nValidation passed: all accusation rows "
            "were marked correctly and no extra files were created."
        )

    return validation_df, unexpected_files_df


validation_df, unexpected_files_df = (
    validate_accusation_transcripts()
)

VALIDATION STATUS


,status,n_games
0,ok,190
1,no_accusations_ok,1


FILES
Expected output files: 190
Actual output files: 190
Unexpected output files: 0

ACCUSATIONS
Expected accusation markers: 3499
Actual accusation markers: 3499

PROBLEMATIC GAMES: 0


,source,session_name,game_key,status,n_dialogue_rows,n_expected_accusations,n_actual_markers,missing_marker_ids,extra_marker_ids,duplicate_marker_ids,exact_content_match,output_path



Validation passed: all accusation rows were marked correctly and no extra files were created.
